# ELECTRA Baseline on PROMISE Expanded

Baseline inspired by *Non Functional Requirements Identification and Classification Using Transfer Learning Model*.

Paper protocol notes used here:

- transfer-learning comparison includes ELECTRA-base and ELECTRA-small;
- preprocessing uses regular expressions, lowercase conversion, stemming, and stopword removal;
- paper reports an 80/20 train-test split, while this repository uses the active 70/15/15 protocol for fair comparison with the other baselines.


In [ ]:
# Kaggle usually has core packages preinstalled. On Colab/local, install missing runtime dependencies.
import os
import sys
import subprocess

if os.path.exists('/kaggle/input'):
    print('Kaggle detected: using preinstalled packages. Enable Internet only if a package/model is missing.')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'pandas', 'torch', 'scikit-learn', 'nltk'])


In [ ]:
# --- Imports ---
import json
import os
import random
import re
import string
import sys
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# --- Config ---
RANDOM_STATE = 42
DATASET_NAME = 'PROMISE expanded'
USE_NFR_ONLY = False  # Set True only if the paper scope explicitly excludes class F.
REMOVE_DIGITS = False  # Earlier project decision: keep measurements such as 99%, 24 hours, 256-bit.
MAX_LENGTH = 128
ELECTRA_MODEL_NAME = 'google/electra-base-discriminator'
# For a lighter ablation, switch to: 'google/electra-small-discriminator'
ELECTRA_MODEL_DIR = None  # Kaggle offline example: '/kaggle/input/electra-base-discriminator/...'

os.environ['WANDB_DISABLED'] = 'true'
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
set_seed(RANDOM_STATE)

# --- NLP resources ---
def setup_nlp_resources():
    try:
        import nltk
        from nltk.corpus import stopwords
        from nltk.stem import PorterStemmer

        nltk.download('stopwords', quiet=True)
        return set(stopwords.words('english')), PorterStemmer(), True
    except Exception as exc:
        for module_name in list(sys.modules):
            if module_name == 'nltk' or module_name.startswith('nltk.'):
                sys.modules.pop(module_name, None)
        print(f"NLTK unavailable ({type(exc).__name__}: {exc}). Using fallback stopwords/stemmer.")

        class IdentityStemmer:
            def stem(self, word):
                return word

        return set(ENGLISH_STOP_WORDS), IdentityStemmer(), False

stop_words, stemmer, NLTK_AVAILABLE = setup_nlp_resources()

# --- Paper-style preprocessing ---
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    if REMOVE_DIGITS:
        text = re.sub(r'\d+', '', text)
    tokens = [word for word in text.split() if word not in stop_words]
    text = ' '.join(stemmer.stem(word) for word in tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
# --- Load PROMISE expanded dataset ---
def find_promise_exp_path():
    candidates = [
        os.path.join('..', 'data', 'exp', 'promise_exp.csv'),
        os.path.join('data', 'exp', 'promise_exp.csv'),
        r'D:\semester4\SWR302\RBL_requirements_classification\rbl-requirements-classification\data\exp\promise_exp.csv',
        '/content/drive/MyDrive/exp/promise_exp.csv',
        '/content/drive/MyDrive/promise_exp.csv',
        '/content/drive/MyDrive/rbl-requirements-classification/data/exp/promise_exp.csv',
    ]

    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'promise_exp.csv' in files:
                return os.path.join(root, 'promise_exp.csv')

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    raise FileNotFoundError('Could not find promise_exp.csv. Expected it under data/exp/promise_exp.csv or a Kaggle/Colab input.')

data_path = find_promise_exp_path()
df = pd.read_csv(data_path)
print(f'Loaded {DATASET_NAME} from: {data_path}')
print('Original shape:', df.shape)
print('Original distribution:', dict(Counter(df['class'])))

required_columns = {'RequirementText', 'class'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing expected columns: {missing_columns}')

if USE_NFR_ONLY:
    df = df[df['class'] != 'F'].copy()
    print('NFR-only mode enabled; removed class F.')

df = df[['RequirementText', 'class']].dropna().reset_index(drop=True)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)
df['cleaned_text'] = df['RequirementText'].apply(clean_text)

print('Active shape:', df.shape)
print('Active distribution:', dict(Counter(df['class'])))

df_train, df_temp = train_test_split(
    df,
    test_size=0.3,
    stratify=df['class'],
    random_state=RANDOM_STATE,
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    stratify=df_temp['class'],
    random_state=RANDOM_STATE,
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print('Train distribution:', dict(Counter(df_train['class'])))
print('Validation distribution:', dict(Counter(df_val['class'])))
print('Test distribution:', dict(Counter(df_test['class'])))


In [ ]:
# --- ELECTRA model loading ---
def is_electra_model_dir(path):
    if not path or not os.path.isdir(path):
        return False
    config_path = os.path.join(path, 'config.json')
    if not os.path.exists(config_path):
        return False
    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            config = json.load(f)
    except Exception:
        return False
    if config.get('model_type') != 'electra':
        return False

    files = set(os.listdir(path))
    has_weights = bool({'pytorch_model.bin', 'model.safetensors'} & files)
    has_tokenizer = bool({'vocab.txt', 'tokenizer.json'} & files)
    return has_weights and has_tokenizer

def find_electra_model_path():
    candidates = []
    if ELECTRA_MODEL_DIR:
        candidates.append(ELECTRA_MODEL_DIR)

    candidates.extend([
        os.path.join('..', 'models', 'electra-base-discriminator'),
        os.path.join('models', 'electra-base-discriminator'),
    ])

    if os.path.exists('/kaggle/input'):
        for root, _, _ in os.walk('/kaggle/input'):
            if is_electra_model_dir(root):
                return root

    for candidate in candidates:
        if is_electra_model_dir(candidate):
            return candidate

    return ELECTRA_MODEL_NAME

electra_model_name_or_path = find_electra_model_path()
electra_local_files_only = os.path.isdir(electra_model_name_or_path)
print(f'Loading ELECTRA from: {electra_model_name_or_path}')

try:
    tokenizer = AutoTokenizer.from_pretrained(
        electra_model_name_or_path,
        local_files_only=electra_local_files_only,
    )
except OSError as exc:
    raise RuntimeError(
        'Cannot load ELECTRA tokenizer. On Kaggle, enable Internet or add a Kaggle model/dataset '
        'containing config.json, tokenizer files, and pytorch_model.bin/model.safetensors; then set ELECTRA_MODEL_DIR.'
    ) from exc


In [ ]:
# --- Encoding and dataset ---
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_train['class'])
y_val = label_encoder.transform(df_val['class'])
y_test = label_encoder.transform(df_test['class'])

train_texts = df_train['cleaned_text'].astype(str).tolist()
val_texts = df_val['cleaned_text'].astype(str).tolist()
test_texts = df_test['cleaned_text'].astype(str).tolist()

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LENGTH)

class RequirementsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = RequirementsDataset(train_encodings, y_train)
val_dataset = RequirementsDataset(val_encodings, y_val)
test_dataset = RequirementsDataset(test_encodings, y_test)


In [ ]:
# --- Metrics ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
    }

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        electra_model_name_or_path,
        num_labels=len(label_encoder.classes_),
        local_files_only=electra_local_files_only,
    )
except OSError as exc:
    raise RuntimeError(
        'Cannot load ELECTRA model weights. On Kaggle, enable Internet or add a local ELECTRA model input; then set ELECTRA_MODEL_DIR.'
    ) from exc

training_args_kwargs = dict(
    output_dir='./results/electra',
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=RANDOM_STATE,
)

try:
    training_args = TrainingArguments(eval_strategy='epoch', **training_args_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy='epoch', **training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()


In [ ]:
# --- Evaluation ---
test_results = trainer.predict(test_dataset)
test_preds = test_results.predictions.argmax(-1)

print('Test metrics:', compute_metrics(test_results))
print('\nClassification report:')
print(classification_report(
    y_test,
    test_preds,
    target_names=label_encoder.classes_,
    zero_division=0,
))

cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ELECTRA Confusion Matrix')
plt.show()

cm_percent = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_percent = np.nan_to_num(cm_percent) * 100
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_percent,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ELECTRA Confusion Matrix (%)')
plt.show()
